# Command-line interface

This example ports a trained classifier of scikit-learn to JavaScript with
the command `porter` of
[sklearn-porter](https://github.com/nok/sklearn-porter), which is available
after the installation. The lines which start with `!` are executed in a
shell.

Running the ported classifier requires `node` for JavaScript.

## Show the usage

The command has the subcommands `show`, `port` and `save`:

In [1]:
!porter -h

usage: porter [-v] [-h] command ...

sklearn-porter CLI v1.0.0

Positional arguments:
  command
    show         Show the supported estimators and programming languages.
    port         Port a trained estimator to a programming language.
    save         Port a trained estimator and save the result.

Options:
  -v, --version  Show the version number and exit.
  -h, --help     Show this help message and exit.

Examples:
  `porter show`
  `porter port model.joblib --language js --template attached`
  `porter save model.joblib --directory /tmp --skip-warnings`

Manuals:
  https://github.com/nok/sklearn-porter
  https://github.com/scikit-learn/scikit-learn


Show the help of a subcommand with `-h`:

In [2]:
!porter port -h

usage: porter port [-l {c,cs,go,java,js,php,ruby}]
                   [-t {attached,combined,exported}] [--class-name CLASS_NAME]
                   [--ignore-preprocessing] [--skip-warnings] [-h]
                   model

The subcommand `port` transpiles a trained estimator (or transformer) to a specific programming language.

Positional arguments:
  model                 Path to an exported estimator.

Options:
  -l {c,cs,go,java,js,php,ruby}, --language {c,cs,go,java,js,php,ruby}
                        The name of the programming language.
  -t {attached,combined,exported}, --template {attached,combined,exported}
                        The name of the template.
  --class-name CLASS_NAME
                        The class name of the ported estimator, which is
                        also the name of the generated files.
  --ignore-preprocessing
                        Port only the final estimator of a pipeline
                        and ignore its preprocessing steps.
  --skip-wa

The subcommand `show` lists the supported estimators and transformers with
the templates of a language:

In [3]:
!porter show -l js

 Estimator                               |   JavaScript
-----------------------------------------+-----------------
 ensemble.AdaBoostClassifier             | ✓ᴀᴾ | ✓ᴄᴾ | ✓ᴇᴾ
 ensemble.ExtraTreesClassifier           | ✓ᴀᴾ | ✓ᴄᴾ | ✓ᴇᴾ
 ensemble.ExtraTreesRegressor            | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 ensemble.GradientBoostingClassifier     | ✓ᴀᴾ | ✓ᴄᴾ | ✓ᴇᴾ
 ensemble.GradientBoostingRegressor      | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 ensemble.HistGradientBoostingClassifier | ✓ᴀᴾ | ✓ᴄᴾ | ✓ᴇᴾ
 ensemble.HistGradientBoostingRegressor  | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 ensemble.IsolationForest                | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 ensemble.RandomForestClassifier         | ✓ᴀᴾ | ✓ᴄᴾ | ✓ᴇᴾ
 ensemble.RandomForestRegressor          | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 linear_model.ARDRegression              | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 linear_model.BayesianRidge              | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 linear_model.ElasticNet                 | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 linear_model.ElasticNetCV               | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 linear_model.HuberRegressor             | ✓ᴀ  | ✓ᴄ  | ✓ᴇ
 linear_m

## Train and save a classifier

To port an estimator (or transformer or pipeline), save it to a file first
(see [model persistence](https://scikit-learn.org/stable/model_persistence.html)).
The files of this example are saved in `/tmp`:

In [4]:
%cd /tmp

/tmp


In [5]:
from joblib import dump
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier

X, y = load_iris(return_X_y=True)
clf = DecisionTreeClassifier(max_depth=2, random_state=0)
clf.fit(X, y)

dump(clf, 'estimator.joblib')

['estimator.joblib']

## Port the classifier

The subcommand `port` prints the source code:

In [6]:
!porter port estimator.joblib -l js -t attached

/*
This file is generated by https://github.com/nok/sklearn-porter/

Estimator:
    DecisionTreeClassifier
    https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

Environment:
    scikit-learn v1.9.1
    sklearn-porter v1.0.0

Usage:
    1. Execute a prediction:
        $ node DecisionTreeClassifier.js feature_1 ... feature_4
    2. Execute the predictions of several samples (one per line):
        $ node DecisionTreeClassifier.js < samples.txt
*/
var DecisionTreeClassifier = function(lefts, rights, thresholds, indices, missingGoToLeft, classes) {

    this.lefts = lefts;
    this.rights = rights;
    this.thresholds = thresholds;
    this.indices = indices;
    this.missingGoToLeft = missingGoToLeft;
    this.classes = classes;

    var _findMax = function(nums) {
        var i = 0, l = nums.length, idx = 0;
        for (; i < l; i++) {
            idx = nums[i] > nums[idx] ? i : idx;
        }
        return idx;
    };

    // Copy the class f

Redirect the output to save the source code to a file:

In [7]:
!porter port estimator.joblib -l js -t attached > estimator.js

Run the ported classifier with the features of a sample:

In [8]:
!node estimator.js 6.3 3.3 6.0 2.5

Predicted class: 2
Probability of class #0 : 0
Probability of class #1 : 0.021739130434782608
Probability of class #2 : 0.9782608695652174


## Save the classifier

The subcommand `save` saves the source code to a file, with the template
`exported` also the model data to a separate JSON file:

In [9]:
!porter save estimator.joblib -l js -t exported --directory /tmp

Saved files:
  /tmp/DecisionTreeClassifier.js
  /tmp/DecisionTreeClassifier.json


## Pass the output to other tools

The output of `port` can be passed to other tools, e.g. to
[terser](https://github.com/terser/terser) (with `npx` of npm), to minify
the code:

```bash
porter port estimator.joblib -l js -t attached | npx terser --compress -o estimator.min.js
```